# S5 · 2차 자료와 자연실험

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 흰 바탕의 글(지금 이것)은 설명이라 실행하지 않습니다. **회색 상자**만 코드이고, 왼쪽의 **▶** 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [S5 · 2차 자료와 자연실험](https://grow.minds.kr/textbooks/css-methods/causal/book/s5-시나리오-2차자료-자연실험.html)

## 1. 준비

아래 **회색 상자 둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. **몇 초 걸리고**, 「경고」 문구가 떠도 정상입니다.
2. 둘째 = 도구와 도우미 함수를 불러옵니다. **「준비 끝」**이 찍히면 됩니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 단절: 행이 사람이 아니라 주다
캠페인 도입 전후로 수준이 이동했나. 4.784 → 5.088.

In [ ]:
ts = load("ts")
pre, post = ts[ts.campaign == 0], ts[ts.campaign == 1]
print(round(pre.wellbeing.mean(), 3), round(post.wellbeing.mean(), 3))
b, se, p, _ = ols(ts.wellbeing, [ts.week, ts.campaign])
print("추세 통제 후 단절:", round(b[2], 3), round(se[2], 3), f"{p[2]:.1e}")

## 3. 위약 검정: 없는 자리에 단절을 세워 본다
도입 전 52주 안에서 가짜 시점을 셋 잡는다. 셋 다 유의하지 않아야 진짜 단절이 산다.

In [ ]:
pre = ts[ts.campaign == 0]
for cut in (13, 26, 39):
    fake = (pre.week >= cut + 1).astype(float)
    b, se, p, _ = ols(pre.wellbeing, [fake])
    print(f"{cut}주 가짜 단절:", round(b[1], 3), round(float(p[1]), 2))

## 4. 시계열의 함정: 잔차가 이웃과 닮았다
독립을 가정한 표준오차는 시계열에서 너무 작다. 잔차 자기상관이 .227 이면 구간을 못 믿는다.

In [ ]:
b, se, p, _ = ols(ts.wellbeing, [ts.week, ts.campaign])
X1 = np.column_stack([np.ones(len(ts)), ts.week, ts.campaign])
resid = ts.wellbeing.values - X1 @ b
print("잔차 자기상관:", round(float(np.corrcoef(resid[:-1], resid[1:])[0, 1]), 3))  # 0.227

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.